In [1]:
# If you have apt in your container:
!apt-get update && apt-get install -y iputils-ping
import os

Get:1 http://packages.osrfoundation.org/gazebo/ubuntu-stable focal InRelease [4279 B]
Get:2 https://packages.hsr.io/ros/ubuntu focal InRelease [6182 B]              
Get:3 http://packages.osrfoundation.org/gazebo/ubuntu-stable focal/main amd64 Packages [134 kB]
Get:4 http://packages.ros.org/ros/ubuntu focal InRelease [4679 B]              
Get:5 http://security.ubuntu.com/ubuntu focal-security InRelease [128 kB]      
Get:6 https://packages.hsr.io/tmc/ubuntu focal InRelease [12.2 kB]             
Get:7 http://archive.ubuntu.com/ubuntu focal InRelease [265 kB]                
Get:8 http://packages.ros.org/ros/ubuntu focal/main amd64 Packages [834 kB]    
Get:9 https://packages.hsr.io/ros/ubuntu focal/main amd64 Packages [176 kB]    
Get:10 https://packages.hsr.io/tmc/ubuntu focal/multiverse amd64 Packages [4744 B]
Get:11 https://packages.hsr.io/tmc/ubuntu focal/main amd64 Packages [2766 B]   
Get:12 http://security.ubuntu.com/ubuntu focal-security/restricted amd64 Packages [4597 kB]
Get

In [4]:
os.system('rosversion -d')
os.system('ifconfig') # ip address of this container
os.system('ping 169.254.4.231')

noetic
eth0: flags=4163<UP,BROADCAST,RUNNING,MULTICAST>  mtu 1500
        inet 172.17.0.2  netmask 255.255.0.0  broadcast 172.17.255.255
        ether 5e:0d:66:ca:eb:db  txqueuelen 0  (Ethernet)
        RX packets 5945  bytes 36931011 (36.9 MB)
        RX errors 0  dropped 0  overruns 0  frame 0
        TX packets 4427  bytes 474444 (474.4 KB)
        TX errors 0  dropped 0 overruns 0  carrier 0  collisions 0

lo: flags=73<UP,LOOPBACK,RUNNING>  mtu 65536
        inet 127.0.0.1  netmask 255.0.0.0
        inet6 ::1  prefixlen 128  scopeid 0x10<host>
        loop  txqueuelen 1000  (Local Loopback)
        RX packets 702  bytes 242963 (242.9 KB)
        RX errors 0  dropped 0  overruns 0  frame 0
        TX packets 702  bytes 242963 (242.9 KB)
        TX errors 0  dropped 0 overruns 0  carrier 0  collisions 0

PING 169.254.4.231 (169.254.4.231) 56(84) bytes of data.
64 bytes from 169.254.4.231: icmp_seq=1 ttl=63 time=0.415 ms
64 bytes from 169.254.4.231: icmp_seq=2 ttl=63 time=0.607 ms
64 

2

In [9]:
import hsrb_interface
import rospy
import sys
import hsrb_interface
from hsrb_interface import geometry

In [18]:
# os.environ['ROS_MASTER_URI'] = 'http://hsrb.local:11311' # alternative is 169.254.4.231
os.environ['ROS_MASTER_URI'] = 'http://169.254.4.231:11311' # alternative is 169.254.4.231
os.environ['ROS_IP'] = '172.17.0.2'  # IP address of this container
print(os.environ['ROS_MASTER_URI'])
print(os.environ['ROS_IP'])

http://169.254.4.231:11311
172.17.0.2


In [19]:
# Move timeout[s]
_MOVE_TIMEOUT=60.0
# Grasp force[N]
_GRASP_FORCE=0.2
# TF name of the OBJECT
_BOTTLE_TF='ar_marker/4'
_TABLE_TF='ar_marker/6'
_HOMEPOS_TF='ar_marker/3'
_TRASH_TF='ar_marker/7'
# TF name of the gripper
_HAND_TF='hand_palm_link'

In [20]:
# Preparation for using the robot functions
robot = hsrb_interface.Robot()
omni_base = robot.get('omni_base')
whole_body = robot.get('whole_body')
gripper = robot.get('gripper')
tts = robot.get('default_tts')

RobotConnectionError: 'NoneType' object has no attribute 'getUri'

In [70]:
# Posture that 0.02[m] front and rotate -1.57 around z-axis of the bottle maker
bottle_to_hand = geometry.pose(z=-0.02, ek=-1.57)
bottle_to_trash = geometry.pose(z=-0.2, ek=-1.57)

# Posture to move the hand 0.1[m] up
hand_up = geometry.pose(x=0.1)

# Posture to move the hand 0.5[m] back
hand_back = geometry.pose(z=-0.5)

# Location of the sofa
sofa_pos = (1.2, 0.4, 1.57)

In [17]:
# Greet
rospy.sleep(3.0)
whole_body.move_to_go()
tts.say('こんにちは. My name is HSR. Nice to meet you. はじめましょう.')
rospy.sleep(5.0)

NameError: name 'whole_body' is not defined

In [72]:
omni_base.pose

[0.1996876919000958, 0.2155823133246017, -0.20140952605014725]

In [73]:
# Transit to initial grasping posture
whole_body.move_to_neutral()
# Look at the hand after the transition
whole_body.looking_hand_constraint = True

In [75]:
try:
     # Transit to initial grasping posture
    whole_body.move_to_go()
    omni_base.go_abs(2.598127270382162, 0.24041103105294316, -0.16538801251765786, 300.0)
    whole_body.move_to_neutral()
    rospy.sleep(1.0)
    omni_base.go_pose(geometry.pose(z=-1.0, ei=3.14, ej=-1.57), 100.0, ref_frame_id='ar_marker/6')
    tts.say('I will scan the bottle')
    omni_base.pose
except:
    tts.say('たすけてください')
    rospy.logerr('fail to init')
    sys.exit()

In [76]:
omni_base.pose #table

[2.7373663498903436, 0.2271743743052535, -0.2532782648420947]

In [77]:
try:
    rospy.sleep(2.0)
    whole_body.move_to_neutral()
    # Look at the hand after the transition
    whole_body.looking_hand_constraint = True
    # Move the hand to front of the bottle
    whole_body.move_end_effector_pose(bottle_to_hand, _BOTTLE_TF)
    # Specify the force to grasp
    gripper.apply_force(_GRASP_FORCE)
    # Wait time for simulator's grasp hack. Not needed on actual robot
    rospy.sleep(2.0)
    # Move the hand up on end effector coordinate
    whole_body.move_end_effector_pose(hand_up, _HAND_TF)
    # Move the hand back on end effector coordinate
    whole_body.move_end_effector_pose(hand_back, _HAND_TF)
    # Transit to initial posture
    whole_body.move_to_neutral()
except:
    tts.say('たすけてください')
    rospy.logerr('fail to init')
    sys.exit()

In [78]:
try:
    omni_base.go_abs(1.6889372569076162, -1.4412442913589325, -1.528230570580849, 300.0)
    tts.say('I will put the bottle into the trash')
    rospy.sleep(5.0)
    whole_body.move_to_neutral()
    #Look at the hand after the transition
    whole_body.looking_hand_constraint = True
    # Move the hand to front of the bottle
    whole_body.move_end_effector_pose(bottle_to_trash, _TRASH_TF)
    gripper.command(1.2)
    omni_base.pose
    whole_body.move_end_effector_pose(hand_back, _HAND_TF)
    whole_body.move_to_go()
    omni_base.go_rel(0.0, 0.0, 2.5, 100.0)

except:
    try:
        omni_base.go_rel(0.0, 0.0, 0.7, 100.0)
        tts.say('I will put the bottle into the trash')
        rospy.sleep(5.0)
        #omni_base.go_rel(1.5, 0.0, 0.0, 100.0)
        whole_body.move_to_neutral()
        #Look at the hand after the transition
        whole_body.looking_hand_constraint = True
        # Move the hand to front of the bottle
        whole_body.move_end_effector_pose(bottle_to_trash, _TRASH_TF)
        gripper.command(1.2)
        whole_body.move_end_effector_pose(hand_back, _HAND_TF)
        whole_body.move_to_go()
        omni_base.go_rel(0.0, 0.0, 2.5, 100.0)
    except:
        omni_base.go_rel(0.0, 0.0, -0.5, 100.0)
        tts.say('I will put the bottle into the trash')
        rospy.sleep(5.0)
        #omni_base.go_rel(1.5, 0.0, 0.0, 100.0)
        whole_body.move_to_neutral()
        #Look at the hand after the transition
        whole_body.looking_hand_constraint = True
        # Move the hand to front of the bottle
        whole_body.move_end_effector_pose(bottle_to_trash, _TRASH_TF)
        gripper.command(1.2)
        whole_body.move_end_effector_pose(hand_back, _HAND_TF)
        whole_body.move_to_go()
        omni_base.go_rel(0.0, 0.0, 5.5, 100.0)
        tts.say('助けてください')
        rospy.logerr('fail to grasp')
        sys.exit()

In [79]:
omni_base.pose

[1.8459472666886823, -1.5672957195346993, 0.9950632838061805]

In [83]:
    #omni_base.go_rel(0.0, 0.0, 1.0, 100.0)
    omni_base.go_abs(-0.000896763500183281, 0.0005740724114742323, -0.0015563745280494801, 300.0)
    whole_body.move_to_go()
    tts.say('おわりました。ありがとうございました')
    whole_body.move_to_joint_positions({'head_tilt_joint': 1.0,'head_tilt_joint': -0.5})
    
    

MobileBaseError: Failed to reach goal ()

In [81]:
try:
    omni_base.go_rel(1.5, 0.1, 0.0, 100.0)
    tts.say('I am going home')
    rospy.sleep(3.0)
    omni_base.go_rel(0.0, 0.0, 1.0, 100.0)
    whole_body.move_to_neutral()
    omni_base.go_rel(2.0, 0.0, 0.0, 100.0)
    whole_body.move_to_go()
    omni_base.go_pose(geometry.pose(z=-0.5, ei=3.14, ej=-1.57), 100.0, ref_frame_id='ar_marker/3')
    rospy.sleep(2.0)
    omni_base.go_rel(0.0, 0.0, 3.5, 100.0)
    whole_body.move_to_go()
    whole_body.move_to_joint_positions({'head_tilt_joint': 1.0,'head_tilt_joint': -0.5})
    tts.say('おわりました。ありがとうございました')
    

except:
    try:
        omni_base.go_rel(0.0, 0.0, -1.0, 100.0)
        whole_body.move_to_neutral()
        omni_base.go_rel(0.5, 0.0, 0.0, 100.0)
        whole_body.move_to_go()
        omni_base.go_pose(geometry.pose(z=-0.5, ei=3.14, ej=-1.57), 100.0, ref_frame_id='ar_marker/3')
        rospy.sleep(2.0)
        omni_base.go_rel(0.0, 0.0, 3.5, 100.0)
        whole_body.move_to_go()
        whole_body.move_to_joint_positions({'head_tilt_joint': 1.0,'head_tilt_joint': -0.5})
    except:
        omni_base.go_rel(0.0, 0.0, 1.0, 100.0)
        whole_body.move_to_neutral()
        omni_base.go_rel(0.5, 0.0, 0.0, 100.0)
        whole_body.move_to_go()
        omni_base.go_pose(geometry.pose(z=-0.5, ei=3.14, ej=-1.57), 100.0, ref_frame_id='ar_marker/3')
        rospy.sleep(2.0)
        omni_base.go_rel(0.0, 0.0, 3.5, 100.0)
        whole_body.move_to_go()
        whole_body.move_to_joint_positions({'head_tilt_joint': 1.0,'head_tilt_joint': -0.5})
    tts.say('おわりました。ありがとうございました')
    tts.say('助けてください')
    rospy.logerr('fail to grasp')
    sys.exit()

In [ ]:
%%bash
rosrun rviz rviz  -d rospack find hsrb_common_launch/config/hsrb_display_full_hsrb.rviz